In [1]:
#import libraries

import scicone
import numpy as np
import pickle
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import io
import scanpy as sc
import anndata
import pyranges
import gseapy as gp

# Set up SCICoNE
install_path = '/cluster/work/bewi/members/andress/pylabs/SCICoNE/build/'
install_path_local = "/home/andress/pylabs/SCICoNE_lab/build/"
temporary_outpath = './'

seed = 42 # for reproducibility

np.random.seed(seed)

# Create SCICoNE object
sci = scicone.SCICoNE(install_path_local, temporary_outpath, verbose=False)

Using binaries at /home/andress/pylabs/SCICoNE_lab/build/


In [2]:


# Define paths
scdna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/cnv/'
scrna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19/'

scdna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/SA501/cnv'
scrna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19'

# Try SSH paths first
try:
    if os.path.exists(scdna_path_ssh) and os.path.exists(scrna_path_ssh):
        scdna_path = scdna_path_ssh
        scrna_path = scrna_path_ssh
    else:
        raise FileNotFoundError("SSH paths are not accessible.")
except FileNotFoundError:
    # Use local paths
    if os.path.exists(scdna_path_local) and os.path.exists(scrna_path_local):
        scdna_path = scdna_path_local
        scrna_path = scrna_path_local
    else:
        raise FileNotFoundError("Neither SSH nor local paths are accessible.")

In [3]:
adatas_path = '/home/andress/pylabs/SCICoNE_lab/rna_imp/adatas'


# Normalize and log transform with a cutoff to avoid negative values
for name in ['clusters_mean', 'clusters_sum', 'clusters_median']:
    cluster_data = anndata.read_h5ad(f'{adatas_path}/adata_{name}.h5ad')
    #sort the data by chromosome
    
    #sc.pp.normalize_total(cluster_data, target_sum=1e4)
    cluster_data.X = np.log1p(cluster_data.X + 1)
    cluster_data.write_h5ad(f'/home/andress/pylabs/SCICoNE_lab/rna_imp/{name}_transformed.h5ad')

    # Inspect transformed data
    print(f"Transformed {name}:", cluster_data.X[:5, :5])

    #print min and max values in transformed data
    print(f"Min and max values in {name}:", cluster_data.X.min(), cluster_data.X.max())



Transformed clusters_mean: [[0.69314718 0.69314718 0.69314718 0.71376647 0.71376647]
 [0.70967648 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.79112759 0.69314718 0.70774598 0.75030559 0.72213472]
 [0.74625701 0.70219702 0.70219702 0.70219702 0.70219702]
 [0.74893854 0.69314718 0.69314718 0.7094077  0.70131049]]
Min and max values in clusters_mean: 0.6931471805599453 6.189358664168741
Transformed clusters_sum: [[0.69314718 0.69314718 0.69314718 1.09861229 1.09861229]
 [1.09861229 0.69314718 0.69314718 0.69314718 0.69314718]
 [2.19722458 0.69314718 1.09861229 1.79175947 1.38629436]
 [2.07944154 1.09861229 1.09861229 1.09861229 1.09861229]
 [2.19722458 0.69314718 0.69314718 1.38629436 1.09861229]]
Min and max values in clusters_sum: 0.6931471805599453 10.067093391134142
Transformed clusters_median: [[0.69314718 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.69314718 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.69314718 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.69314718 0

In [ ]:
# Correctly compute and save stop indexes for each chromosome
adata = anndata.read_h5ad(f'{temporary_outpath}/adata_mt_filtered.h5ad')

# Ensure chromosome names are clean
df_exp_annotations_2 = df_exp_annotations_sorted.copy()
df_exp_annotations_2['Chromosome'] = df_exp_annotations_2['Chromosome'].astype(str).str.strip()

# Filter for standard chromosomes
standard_chromosomes = [str(i) for i in range(1, 23)] + ["X", "Y"]
filtered_df = df_exp_annotations_2[df_exp_annotations_2['Chromosome'].isin(standard_chromosomes)]

# Find the max end position for each chromosome
idx_max = filtered_df.groupby('Chromosome')['End'].idxmax()
chromosome_stops_df = filtered_df.loc[idx_max, ['Chromosome', 'End', 'ensembl_gene_id']].reset_index(drop=True)

# Retrieve the column position of the max end position in the original adata object
chromosome_stops_df['column_position'] = chromosome_stops_df['ensembl_gene_id'].apply(
    lambda gene_id: int(np.where(adata.var['ensembl_gene_id'] == gene_id)[0][0]) if gene_id in adata.var['ensembl_gene_id'].values else -1
)

# Sort the list by chromosome order
chromosome_order = {str(i): i for i in range(1, 23)}
chromosome_order.update({'X': 23, 'Y': 24})
chromosome_stops_df['chr_order'] = chromosome_stops_df['Chromosome'].map(chromosome_order)
chromosome_stops_df = chromosome_stops_df.sort_values(by='chr_order').drop('chr_order', axis=1).reset_index(drop=True)

# Extract the list of stop indexes
stop_indexes = [0] + chromosome_stops_df['column_position'].tolist() + [adata.shape[1] - 1]
print("Stop indexes:", stop_indexes)

# Save the stop indexes to a JSON file
import json
with open(os.path.join(temporary_outpath, 'stop_indexes.json'), 'w') as f:
    json.dump(stop_indexes, f)

print("Stop indexes saved to JSON.")

In [8]:
def extract_chromosome_stops_dict(adata, temporary_outpath):
    """
    Extracts the stop positions (max end positions) for each chromosome from an AnnData object
    and saves them as a dictionary with 'Start', 'End', and chromosome numbers as keys.

    :param adata: AnnData object containing gene annotations in `adata.var`.
    :param temporary_outpath: Path to save the stop indexes JSON file.
    :return: Sorted dictionary with 'Start', 'End', and chromosome numbers as keys and stop positions as values.
    """
    # Ensure chromosome names are clean
    adata.var['Chromosome'] = adata.var['Chromosome'].astype(str).str.strip()

    # Filter for standard chromosomes
    standard_chromosomes = [str(i) for i in range(1, 23)] + ["X", "Y"]
    filtered_var = adata.var[adata.var['Chromosome'].isin(standard_chromosomes)]

    # Sort chromosomes
    sorted_chromosomes = sort_chromosomes(filtered_var['Chromosome'].unique())

    # Find the max end position for each chromosome
    chromosome_stops = {}
    for chromosome in sorted_chromosomes:
        chromosome_data = filtered_var[filtered_var['Chromosome'] == chromosome]
        if not chromosome_data.empty:
            max_end_idx = chromosome_data['End'].idxmax()
            max_gene_id = chromosome_data.loc[max_end_idx, 'ensembl_gene_id']
            column_position = int(np.where(adata.var['ensembl_gene_id'] == max_gene_id)[0][0])
            chromosome_stops[chromosome] = column_position

    # Sort the dictionary by chromosome order
    chromosome_order = {str(i): i for i in range(1, 23)}
    chromosome_order.update({'X': 23, 'Y': 24, 'Start': 0, 'End': 25})
    chromosome_stops = dict(sorted(chromosome_stops.items(), key=lambda x: chromosome_order.get(x[0], float('inf'))))

    # Save the chromosome stops dictionary to a JSON file
    with open(os.path.join(temporary_outpath, 'chromosome_stops.json'), 'w') as f:
        json.dump(chromosome_stops, f)

    print("Chromosome stops saved to JSON:", chromosome_stops)
    return chromosome_stops

def sort_chromosomes(chromosome_list):
    """
    Sorts a list of unordered chromosome names.
    :param chromosome_list: list of unordered characters denoting chromosomes '1', '2', ..., 'X', 'Y'.
    """
    # Replace X and Y with 23 and 24
    sorted_chromosome_list = np.array(chromosome_list)
    sorted_chromosome_list[np.where(sorted_chromosome_list == "X")[0]] = 23
    sorted_chromosome_list[np.where(sorted_chromosome_list == "Y")[0]] = 24

    # Convert everything to integer
    sorted_chromosome_list = sorted_chromosome_list.astype(int)

    # Sort
    sorted_chromosome_list = np.sort(sorted_chromosome_list)

    # Convert back to string
    sorted_chromosome_list = sorted_chromosome_list.astype(str)
    sorted_chromosome_list[np.where(sorted_chromosome_list == "23")[0]] = "X"
    sorted_chromosome_list[np.where(sorted_chromosome_list == "24")[0]] = "Y"

    return sorted_chromosome_list

In [18]:
adata = anndata.read_h5ad(f'{temporary_outpath}/adatas/adata_leiden.h5ad')
print(adata.shape)
annot = sc.queries.biomart_annotations(
        "hsapiens",
        ["ensembl_gene_id", "start_position", "end_position", "chromosome_name"],
    ).set_index("ensembl_gene_id")

annot = annot.reset_index()
annot = annot.rename(columns={'start_position':'Start', 'end_position': 'End', 'chromosome_name': 'Chromosome'})

gr_annotations = pyranges.from_dict(annot.to_dict())

gr_annotations.head()
# Get gene coordinates

df_annotations = gr_annotations.df.set_index('ensembl_gene_id')

df_exp_annotations = df_annotations.loc[df_annotations.index.intersection(adata.var['ensembl_gene_id'])]\
                        .reset_index().rename(columns={'index':'ensembl_gene_id'})

adata.var_names = adata.var['ensembl_gene_id'].values

adata = adata[:,df_exp_annotations['ensembl_gene_id']]

df_exp_annotations_sorted = df_exp_annotations.sort_values(by=['Chromosome', 'End'])

df_exp_annotations_sorted.head()

# merged_var = adata.var.merge(
#     df_annotations[['Chromosome', 'Start', 'End']],
#     how='left',
#     left_on='ensembl_gene_id',
#     right_index=True
# )

# Ensure the index matches the original var index
# merged_var.index = adata.var.index
# adata.var = merged_var
print('you are here')
adata

(1537, 17153)
you are here


View of AnnData object with n_obs × n_vars = 1537 × 17153
    obs: 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden'
    var: 'ensembl_gene_id', 'gene_id', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'Chromosome', 'End'
    uns: 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [19]:
#print adata shape
print("Shape of adata:", adata.shape)

Shape of adata: (1537, 17153)


In [21]:
import json
extract_chromosome_stops_dict(adata, temporary_outpath)

Chromosome stops saved to JSON: {np.str_('1'): 218, np.str_('2'): 2859, np.str_('3'): 3508, np.str_('4'): 4457, np.str_('5'): 5335, np.str_('6'): 6090, np.str_('7'): 6370, np.str_('8'): 7179, np.str_('9'): 8215, np.str_('10'): 8612, np.str_('11'): 9783, np.str_('12'): 10119, np.str_('13'): 11024, np.str_('14'): 11571, np.str_('15'): 12091, np.str_('16'): 12519, np.str_('17'): 13423, np.str_('18'): 14172, np.str_('19'): 15474, np.str_('20'): 15845, np.str_('21'): 16110, np.str_('22'): 16505, np.str_('X'): 16700, np.str_('Y'): 17151}


{np.str_('1'): 218,
 np.str_('2'): 2859,
 np.str_('3'): 3508,
 np.str_('4'): 4457,
 np.str_('5'): 5335,
 np.str_('6'): 6090,
 np.str_('7'): 6370,
 np.str_('8'): 7179,
 np.str_('9'): 8215,
 np.str_('10'): 8612,
 np.str_('11'): 9783,
 np.str_('12'): 10119,
 np.str_('13'): 11024,
 np.str_('14'): 11571,
 np.str_('15'): 12091,
 np.str_('16'): 12519,
 np.str_('17'): 13423,
 np.str_('18'): 14172,
 np.str_('19'): 15474,
 np.str_('20'): 15845,
 np.str_('21'): 16110,
 np.str_('22'): 16505,
 np.str_('X'): 16700,
 np.str_('Y'): 17151}